In [ ]:
"""
{
    "GPT-OSS-120B": {
        "secret":        "NVIDIA_API_KEY",
        "base_url":      NIM_BASE_URL,
        "model_id":      "openai/gpt-oss-120b",
        "has_reasoning": True,
    },
    "Llama-3.1-70B": {
        "secret": "NVIDIA_API_KEY",
        "base_url": NIM_BASE_URL,
        "model_id": "meta/llama-3.1-70b-instruct",
        "has_reasoning": False,
    },
    "mistralai-2-123b": {
        "secret": "NVIDIA_API_KEY",
        "base_url": NIM_BASE_URL,
        "model_id": "mistralai/devstral-2-123b-instruct-2512",
        "has_reasoning": False,
        "type": "api"
    },
    "qwen3-80b": {
        "secret": "NVIDIA_API_KEY",
        "base_url": NIM_BASE_URL,
        "model_id": "qwen/qwen3-next-80b-a3b-instruct",
        "has_reasoning": False,
        "type": "api"
    }
}

"""

In [2]:
import shutil

src = "/kaggle/input/datasets/data-augment44/checkpoint_qwen (2).json"
dst = "/kaggle/working/checkpoint_qwen.json"

shutil.copy(src, dst)

'/kaggle/working/checkpoint_qwen.json'

In [ ]:
import os, sys, re, json, time, random, subprocess
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm import tqdm

def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip("openai>=1.30.0", "tenacity", "tqdm", "ushuffle")

from openai import OpenAI, RateLimitError, APIStatusError
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import ushuffle

WORK_DIR          = "/kaggle/working"
OUT_CSV           = os.path.join(WORK_DIR, "dart_eval_synthetic.csv")

N_TFS             = 50    # number of TF motifs to generate
N_BACKGROUNDS     = 100   # background sequences per TF
SEQ_LEN           = 350   # final sequence length in bp
BACKGROUNDS_BATCH = 25    # backgrounds requested per API call
API_DELAY         = 1.0   # seconds between calls

NIM_BASE_URL      = "https://integrate.api.nvidia.com/v1"
MODEL_ID          = "qwen/qwen3-next-80b-a3b-instruct" #modify model id and name until data augmentation is complete using all 4 llms
API_KEY_SECRET    = "NVIDIA_API_KEY"
model_name        = "qwen"


def load_api_key():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(API_KEY_SECRET)
    except Exception:
        return os.environ.get(API_KEY_SECRET, "")


@retry(
    retry=retry_if_exception_type((RateLimitError, APIStatusError)),
    wait=wait_exponential(multiplier=2, min=5, max=120),
    stop=stop_after_attempt(8),
)
def call_llm(client, prompt):
    resp = client.chat.completions.create(
        model=MODEL_ID,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=4000,
        temperature=0.7,
        top_p=0.95,
    )
    return (resp.choices[0].message.content or "").strip()

CHECKPOINT = os.path.join(WORK_DIR, f"checkpoint_{model_name}.json")

def load_checkpoint():
    if os.path.exists(CHECKPOINT):
        with open(CHECKPOINT) as f:
            data = json.load(f)
        print(f"resuming from checkpoint — {len(data)} TFs already done")
        return data
    return {}

def save_checkpoint(data):
    tmp = CHECKPOINT + ".tmp"
    with open(tmp, "w") as f:
        json.dump(data, f)
    os.replace(tmp, CHECKPOINT)



def clean_dna(seq):
    """Strip everything that isn't A/C/G/T and uppercase."""
    return re.sub(r"[^ACGTacgt]", "", seq).upper()


def pad_or_trim(seq, length):
    """Force a sequence to exactly `length` bases."""
    if len(seq) >= length:
        return seq[:length]
    pool = list(seq)
    while len(seq) < length:
        seq += random.choice(pool)
    return seq[:length]


def reverse_complement(seq):
    table = str.maketrans("ACGT", "TGCA")
    return seq.translate(table)[::-1]


def _python_dinuc_shuffle(seq):
    edges = defaultdict(list)
    for i in range(len(seq) - 1):
        edges[seq[i]].append(seq[i + 1])

    for base in edges:
        random.shuffle(edges[base])

    result = [seq[0]]
    current = seq[0]
    while edges[current]:
        nxt = edges[current].pop(0)
        result.append(nxt)
        current = nxt

    return pad_or_trim("".join(result), len(seq))


def dinucleotide_shuffle(seq):
    try:
        return ushuffle.shuffle(seq.encode(), 2).decode()
    except Exception:
        return _python_dinuc_shuffle(seq)


def shuffle_motif(motif):
    bases = list(motif)
    random.shuffle(bases)
    return "".join(bases)


def insert_at_center(background, motif):
    """Replace the central bases of `background` with `motif`."""
    start = SEQ_LEN // 2 - len(motif) // 2
    return background[:start] + motif + background[start + len(motif):]



def generate_motif(client):
    """Ask the LLM for a single TF binding motif (6-12 bp, ACGT only)."""
    prompt = (
        "You are generating data for a DNA benchmark.\n\n"
        "Give me ONE transcription factor binding motif between 6 and 12 base pairs.\n"
        "Rules:\n"
        "- Use only A, C, G, T\n"
        "- The motif should resemble a real TF binding site "
        "(examples: TGAGTCA, CACGTG, GGGGCGGGGC)\n"
        "- Output ONLY the motif sequence, nothing else\n"
        "- No spaces, no numbers, no explanation"
    )
    raw = call_llm(client, prompt)
    for token in raw.split():
        motif = clean_dna(token)
        if 6 <= len(motif) <= 12:
            return motif
    cleaned = clean_dna(raw)
    return cleaned[:8] if len(cleaned) >= 6 else "TGAGTCA"


def generate_backgrounds(client, n):
    """
    Generate `n` background sequences via the LLM, then dinucleotide-shuffle
    each one before returning.
    """
    backgrounds = []

    while len(backgrounds) < n:
        need = min(BACKGROUNDS_BATCH, n - len(backgrounds))

        prompt = (
            f"You are generating data for a DNA benchmark.\n\n"
            f"Give me exactly {need} regulatory DNA background sequences.\n"
            "Rules:\n"
            f"- Each sequence must be exactly {SEQ_LEN} base pairs long\n"
            "- Use only A, C, G, T\n"
            "- GC content between 40% and 60%\n"
            "- Realistic dinucleotide transitions, no long homopolymers\n"
            "- One sequence per line, nothing else — no labels, no numbers"
        )

        raw = call_llm(client, prompt)

        for line in raw.splitlines():
            seq = clean_dna(line)
            if len(seq) < SEQ_LEN // 2:
                continue
            seq = pad_or_trim(seq, SEQ_LEN)
            seq = dinucleotide_shuffle(seq)
            backgrounds.append(seq)
            if len(backgrounds) == n:
                break

        time.sleep(API_DELAY)

    return backgrounds



def build_sequences(motif, backgrounds):

    neg_motif = shuffle_motif(motif)
    rows = []

    for bg in backgrounds:
        pos_fwd = insert_at_center(bg, motif)
        neg_fwd = insert_at_center(bg, neg_motif)

        rows.append({"sequence": pos_fwd,                     "label": "forward_true"})
        rows.append({"sequence": reverse_complement(pos_fwd), "label": "reverse_true"})
        rows.append({"sequence": neg_fwd,                     "label": "forward_shuffled"})
        rows.append({"sequence": reverse_complement(neg_fwd), "label": "reverse_shuffled"})

    return rows



def main():
    random.seed(42)
    np.random.seed(42)
    os.makedirs(WORK_DIR, exist_ok=True)

    api_key = load_api_key()
    if not api_key:
        raise RuntimeError("No API key found — set NVIDIA_API_KEY in Kaggle secrets")

    client = OpenAI(api_key=api_key, base_url=NIM_BASE_URL)

    checkpoint = load_checkpoint()
    all_rows = []

    for rows in checkpoint.values():
        all_rows.extend(rows)

    for i in tqdm(range(N_TFS), desc="TFs"):
        key = f"tf_{i}"
        if key in checkpoint:
            continue

        motif = generate_motif(client)
        time.sleep(API_DELAY)

        backgrounds = generate_backgrounds(client, N_BACKGROUNDS)

        rows = build_sequences(motif, backgrounds)
        for r in rows:
            r["tf_name"] = key
            r["motif"]   = motif

        all_rows.extend(rows)
        checkpoint[key] = rows
        save_checkpoint(checkpoint)

    df = pd.DataFrame(all_rows)[["sequence", "label", "tf_name", "motif"]]
    df.to_csv(OUT_CSV, index=False)

    print(f"\n{len(df):,} sequences saved to {OUT_CSV}")
    print(df["label"].value_counts().to_string())


if __name__ == "__main__":
    main()